# Semi-annual Model Inventory Attestation

**Step 1** – Clean master copy: keeps only the nine attestation tabs, converts formulas to values, removes external links, adds a `Metadata` tab.

**Step 2** – One Excel file per owner (PLC Model Owner / PLC DQM Owner / PMA Owner), filtered via `Model ID`, `DQM ID` and `PMA ID`. A person who owns models, DQMs and PMAs gets one file with all relevant tabs; tab groups they own nothing in are dropped.

Only edit the **Configuration** cell, then run all cells top to bottom (Kernel ▸ Restart & Run All).

Requires: `openpyxl >= 3.1`, `pandas`.

## 1. Configuration

In [ ]:
from pathlib import Path

# --- Paths -------------------------------------------------------------
INPUT_FILE = Path(r"C:\MRM\Inventory\MRM_Model_Inventory.xlsx")   # current inventory
OUTPUT_DIR = Path(r"C:\MRM\Attestation\2026_H2")                  # created if missing

# --- Tabs to include in the attestation (in this order) ---------------
ATTESTATION_SHEETS = [
    "Model Inventory", "Model Limitation", "Model Validations",
    "DQM Inventory",   "DQM Limitation",   "DQM Reviews",
    "PMA Proposals",   "PMA Limitation",   "PMA Validations",
]

# --- How owners are linked to each tab group --------------------------
GROUPS = {
    "Model": {"inventory_sheet": "Model Inventory", "owner_col": "PLC Model Owner", "id_col": "Model ID",
              "sheets": ["Model Inventory", "Model Limitation", "Model Validations"]},
    "DQM":   {"inventory_sheet": "DQM Inventory",   "owner_col": "PLC DQM Owner",   "id_col": "DQM ID",
              "sheets": ["DQM Inventory", "DQM Limitation", "DQM Reviews"]},
    "PMA":   {"inventory_sheet": "PMA Proposals",   "owner_col": "PMA Owner",       "id_col": "PMA ID",
              "sheets": ["PMA Proposals", "PMA Limitation", "PMA Validations"]},
}

# If a tab names its ID column differently, map it here, e.g. {"Model Validations": "Model Ref"}
ID_COLUMN_OVERRIDES = {}

# If one owner cell can hold several names, give the separator(s) as a regex, e.g. r"[;/\n]"
# Leave as None to treat each cell as a single name.
OWNER_SEPARATOR = None

REMOVE_HYPERLINKS  = True   # strip cell hyperlinks (e.g. links to SharePoint docs)
HEADER_SEARCH_ROWS = 20     # header row is detected automatically within the first N rows

## 2. Helper functions (no need to edit)

In [ ]:
import re
import getpass
from datetime import datetime

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter, range_boundaries
from openpyxl.worksheet.formula import ArrayFormula, DataTableFormula

RUN_TS = datetime.now()
STAMP = RUN_TS.strftime("%Y%m%d")
KEPT = set(ATTESTATION_SHEETS)


def norm_text(v):
    """Case- and whitespace-insensitive key, so 'John  Doe' == 'john doe'."""
    return " ".join(str(v).split()).casefold() if v is not None else ""


def norm_id(v):
    """Makes 101, 101.0 and '101' match each other."""
    if v is None:
        return ""
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return str(v).strip().casefold()


def split_owners(v):
    if v is None or str(v).strip() == "":
        return []
    parts = re.split(OWNER_SEPARATOR, str(v)) if OWNER_SEPARATOR else [str(v)]
    return [" ".join(p.split()) for p in parts if p.strip()]


def id_col_for(sheet, group_cfg):
    return ID_COLUMN_OVERRIDES.get(sheet, group_cfg["id_col"])


def find_header(ws, col_name):
    """Returns (header_row, column_index) of col_name within the first HEADER_SEARCH_ROWS rows."""
    target = norm_text(col_name)
    for row in ws.iter_rows(min_row=1, max_row=min(HEADER_SEARCH_ROWS, ws.max_row)):
        for c in row:
            if norm_text(c.value) == target:
                return c.row, c.column
    raise ValueError(f"Column '{col_name}' not found in the first {HEADER_SEARCH_ROWS} rows of tab '{ws.title}'.")


def formula_refs_ok(formula, valid_names, kept):
    """False if a formula points at a removed tab, an external file or a removed named range."""
    if not formula:
        return True
    f = str(formula).lstrip("=").strip()
    if f.startswith('"'):
        return True                                   # literal list, e.g. "Yes,No"
    if "[" in f:
        return False                                  # external workbook
    refs = re.findall(r"(?:'((?:[^']|'')+)'|([A-Za-z0-9_\.]+))!", f)
    for quoted, plain in refs:
        if (quoted or plain).replace("''", "'") not in kept:
            return False
    if not refs and re.fullmatch(r"[A-Za-z_\\][\w\.]*", f) and not re.fullmatch(r"\$?[A-Za-z]{1,3}\$?\d+", f):
        return f.casefold() in valid_names            # bare named range
    return True


def clean_references(wb):
    """Drops named ranges and dropdown lists that point at tabs no longer in the workbook or at other files."""
    kept = set(wb.sheetnames)
    for name, dn in list(wb.defined_names.items()):
        if not formula_refs_ok(dn.attr_text, set(), kept):
            del wb.defined_names[name]
    valid_names = {n.casefold() for n in wb.defined_names}
    for ws in wb.worksheets:
        for name, dn in list(ws.defined_names.items()):
            if not formula_refs_ok(dn.attr_text, valid_names, kept):
                del ws.defined_names[name]
        ws.data_validations.dataValidation = [
            dv for dv in ws.data_validations.dataValidation
            if formula_refs_ok(dv.formula1, valid_names, kept) and formula_refs_ok(dv.formula2, valid_names, kept)
        ]


def safe_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "", name).strip().replace(" ", "_")[:80] or "Unknown"


def reset_tab_selection(wb):
    for ws in wb.worksheets:
        ws.sheet_view.tabSelected = False
    wb.active = 0
    wb.worksheets[0].sheet_view.tabSelected = True


def write_metadata(ws, rows):
    ws.append(["Field", "Value"])
    for c in ws[1]:
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="1F4E78")
    for r in rows:
        ws.append(list(r))
    ws.column_dimensions["A"].width = 32
    ws.column_dimensions["B"].width = 90

## 3. Step 1 – Create the clean master copy

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

wb      = load_workbook(INPUT_FILE, keep_links=False)                  # formats + formulas
wb_vals = load_workbook(INPUT_FILE, keep_links=False, data_only=True)  # last values Excel calculated

missing = [s for s in ATTESTATION_SHEETS if s not in wb.sheetnames]
if missing:
    raise ValueError(f"Tabs not found in {INPUT_FILE.name}: {missing}")

# 1a. Drop every other tab and put the nine in the configured order
for name in list(wb.sheetnames):
    if name not in KEPT:
        del wb[name]
wb._sheets.sort(key=lambda ws: ATTESTATION_SHEETS.index(ws.title))

# 1b. Formulas -> values (and strip hyperlinks)
n_formulas, no_cached = 0, []
for ws in wb.worksheets:
    vals = wb_vals[ws.title]
    for row in ws.iter_rows():
        for c in row:
            if c.data_type == "f" or isinstance(c.value, (ArrayFormula, DataTableFormula)):
                c.value = vals[c.coordinate].value
                n_formulas += 1
                if c.value is None:
                    no_cached.append(f"{ws.title}!{c.coordinate}")
            if REMOVE_HYPERLINKS and getattr(c, "hyperlink", None) is not None:
                c.hyperlink = None

# 1c. Remove named ranges / dropdowns that would now point at removed tabs or other files
clean_references(wb)

# 1d. Metadata tab
write_metadata(wb.create_sheet("Metadata"), [
    ("Date of copy",                 RUN_TS.strftime("%d/%m/%Y %H:%M")),
    ("File name copied",             INPUT_FILE.name),
    ("Full source path",             str(INPUT_FILE.resolve())),
    ("Tabs included",                ", ".join(ATTESTATION_SHEETS)),
    ("Formulas converted to values", n_formulas),
    ("Prepared by",                  getpass.getuser()),
])

reset_tab_selection(wb)
MASTER_FILE = OUTPUT_DIR / f"Inventory_Attestation_MASTER_{STAMP}.xlsx"
wb.save(MASTER_FILE)

print(f"Master copy saved: {MASTER_FILE}")
print(f"Formulas converted to values: {n_formulas}")
if no_cached:
    print(f"WARNING: {len(no_cached)} formula cells had no stored value (blank result, or file not "
          f"recalculated/saved in Excel). First few: {no_cached[:10]}")

## 4. Step 2a – Build the owner list and check the data
Review the output of this cell before generating files: unassigned items and orphan IDs are worth fixing first.

In [ ]:
wbm = load_workbook(MASTER_FILE)

owners = {}        # normalised name -> {"Owner": display name, "Model": {...}, "DQM": {...}, "PMA": {...}}
unassigned = []    # items with an ID but no owner
for group, cfg in GROUPS.items():
    ws = wbm[cfg["inventory_sheet"]]
    hdr, own_col = find_header(ws, cfg["owner_col"])
    _,   id_col  = find_header(ws, id_col_for(cfg["inventory_sheet"], cfg))
    for r in range(hdr + 1, ws.max_row + 1):
        item_id = norm_id(ws.cell(r, id_col).value)
        if not item_id:
            continue
        names = split_owners(ws.cell(r, own_col).value)
        if not names:
            unassigned.append({"Group": group, "ID": ws.cell(r, id_col).value, "Tab": ws.title, "Row": r})
        for n in names:
            entry = owners.setdefault(norm_text(n), {"Owner": n, **{g: set() for g in GROUPS}})
            entry[group].add(item_id)

# IDs in limitation / validation tabs that are not in the inventory tab (they'd go to nobody)
all_ids = {g: set().union(*(o[g] for o in owners.values())) if owners else set() for g in GROUPS}
orphans = []
for group, cfg in GROUPS.items():
    inv_ws = wbm[cfg["inventory_sheet"]]
    hdr, id_col = find_header(inv_ws, id_col_for(cfg["inventory_sheet"], cfg))
    inventory_ids = {norm_id(inv_ws.cell(r, id_col).value) for r in range(hdr + 1, inv_ws.max_row + 1)}
    for sheet in cfg["sheets"][1:]:
        ws = wbm[sheet]
        hdr, id_col = find_header(ws, id_col_for(sheet, cfg))
        for r in range(hdr + 1, ws.max_row + 1):
            v = norm_id(ws.cell(r, id_col).value)
            if v and v not in inventory_ids:
                orphans.append({"Group": group, "Tab": sheet, "Row": r, "ID": ws.cell(r, id_col).value})

summary = pd.DataFrame(
    [{"Owner": o["Owner"], **{f"# {g}s": len(o[g]) for g in GROUPS}} for o in owners.values()]
).sort_values("Owner").reset_index(drop=True)

print(f"{len(owners)} distinct owners found")
display(summary)
if unassigned:
    print(f"\n{len(unassigned)} item(s) with no owner (no file will include them):")
    display(pd.DataFrame(unassigned))
if orphans:
    print(f"\n{len(orphans)} row(s) whose ID is not in the inventory tab (no file will include them):")
    display(pd.DataFrame(orphans))

## 5. Step 2b – Generate one file per owner

In [ ]:
def filter_sheet(ws, id_col_name, keep_ids):
    """Deletes every data row whose ID is not in keep_ids, keeping formats, row heights, tables and filters intact."""
    hdr, id_col = find_header(ws, id_col_name)
    last = ws.max_row

    # merged cells below the header would be corrupted by row deletion
    for mr in list(ws.merged_cells.ranges):
        if mr.min_row > hdr:
            ws.unmerge_cells(str(mr))

    rows = range(hdr + 1, last + 1)
    keep = [r for r in rows if norm_id(ws.cell(r, id_col).value) in keep_ids]
    drop = [r for r in rows if r not in set(keep)]
    dims = [(ws.row_dimensions[r].height, ws.row_dimensions[r].hidden) for r in keep]

    # delete contiguous blocks, bottom-up
    blocks, start, prev = [], None, None
    for r in drop:
        if start is None:
            start = prev = r
        elif r == prev + 1:
            prev = r
        else:
            blocks.append((start, prev)); start = prev = r
    if start is not None:
        blocks.append((start, prev))
    for s, e in reversed(blocks):
        ws.delete_rows(s, e - s + 1)

    # re-apply row heights to the rows that moved up
    new_last = hdr + len(keep)
    for i, (h, hidden) in enumerate(dims):
        rd = ws.row_dimensions[hdr + 1 + i]
        rd.height, rd.hidden = h, hidden
    for r in range(new_last + 1, last + 1):
        if r in ws.row_dimensions:
            del ws.row_dimensions[r]

    # shrink Excel tables and autofilters to the new size
    for t in ws.tables.values():
        c1, r1, c2, r2 = range_boundaries(t.ref)
        if r1 <= hdr <= r2:
            t.ref = f"{get_column_letter(c1)}{r1}:{get_column_letter(c2)}{max(new_last, r1 + 1)}"
            if t.autoFilter is not None:
                t.autoFilter.ref = t.ref
    if ws.auto_filter.ref:
        c1, r1, c2, r2 = range_boundaries(ws.auto_filter.ref)
        ws.auto_filter.ref = f"{get_column_letter(c1)}{r1}:{get_column_letter(c2)}{max(new_last, r1 + 1)}"
    return len(keep)


OWNER_DIR = OUTPUT_DIR / "Owner files"
OWNER_DIR.mkdir(exist_ok=True)

log, used_names = [], set()
for key, o in sorted(owners.items()):
    wbo = load_workbook(MASTER_FILE)
    row_counts = {}
    for group, cfg in GROUPS.items():
        for sheet in cfg["sheets"]:
            if o[group]:
                row_counts[sheet] = filter_sheet(wbo[sheet], id_col_for(sheet, cfg), o[group])
            else:
                del wbo[sheet]                      # owner has nothing in this group

    clean_references(wbo)
    meta = wbo["Metadata"]
    for row in meta.iter_rows(min_col=1, max_col=2):
        if row[0].value == "Tabs included":
            row[1].value = ", ".join(s for s in wbo.sheetnames if s != "Metadata")
    meta.append(["Owner", o["Owner"]])
    for g in GROUPS:
        meta.append([f"{g} IDs included", ", ".join(sorted(o[g])).upper() or "None"])
    reset_tab_selection(wbo)

    fname = f"Inventory_Attestation_{safe_filename(o['Owner'])}_{STAMP}"
    n = 2
    while fname.casefold() in used_names:            # avoid two owners overwriting one file
        fname = f"Inventory_Attestation_{safe_filename(o['Owner'])}_{n}_{STAMP}"; n += 1
    used_names.add(fname.casefold())
    path = OWNER_DIR / f"{fname}.xlsx"
    wbo.save(path)

    log.append({"Owner": o["Owner"], **{f"# {g}s": len(o[g]) for g in GROUPS},
                "Tabs": len(wbo.sheetnames) - 1, "File": path.name,
                "Date sent": None, "Response received": None, "Changes requested": None})
    print(f"✓ {o['Owner']:<35} {path.name}")

log_df = pd.DataFrame(log)
LOG_FILE = OUTPUT_DIR / f"Attestation_Distribution_Log_{STAMP}.xlsx"
with pd.ExcelWriter(LOG_FILE) as xw:
    log_df.to_excel(xw, sheet_name="Distribution log", index=False)
    if unassigned:
        pd.DataFrame(unassigned).to_excel(xw, sheet_name="No owner", index=False)
    if orphans:
        pd.DataFrame(orphans).to_excel(xw, sheet_name="Orphan IDs", index=False)

print(f"\n{len(log)} owner files saved in: {OWNER_DIR}")
print(f"Distribution log: {LOG_FILE}")